# 09. Naive News Letter Delivery

## 09.01. User Stories Are Not Set In Stone

##### 09.01.0.1. Skimming: What did you notice and why? Any Questinos

_**What?**_  
It is a great habbit to revisit user stories and refine them as we get more precise about implementation.  
Author makes note of how we now have `confirmed` subscribers and subcribers `pending_confirmation`.  

Now that we are building out our newsletter email delivery we use more precise language to articulate that we only want  
to deliver send out emails to only confirmed subscribers.

_**Why?**_  
I've encountered this in my experience how a user story gets more clear implemetation occurs.

This looks a little different pre-AI and now currently with AI, because with AI now we can dig deeper and refine a PRD as we  
flesh out supporting documents like the ADRs, Data Model, API spec, testing strategy, ci-cd documentation and project structure.  
When this is done before writing any code, there's a lot of clarify to agents, developers and product ownwers about what is built  
from a high level to the granular details.

_**Questions**_  
I wonder how the appropriate amount of time to spend on refining a PRD especially now in the age of AI before writing any code.   
Because before it was usually the case that a lot of things would get discovered while development was happening. Because of the  
speed at which we can generate code now it seems that you can have moved too quickly before refining the implementation. Sometimes  
by the time you're revisiting the work, it feels like you have to start from scratch.

I guess the question is it more worth it currently to do the documentation work upfront, identify the core of a product and refine  
that, so that the first iteration is surgical and extensible? In other words fleshing our a core that is both backward and forward compatible  
with a docuumentation first approach that allows one to clearly define (as best as possible) what the core should look like.

Here it means then that humans can become the bottleneck as well, either going too fast to truly define the core, or too slow to leverage  
what can be done with AI. Something optimal is somewhere inbetween the two. One thing is for sure, clarity is super benefitial for humans   
but now even more when couple with AI.

##### 09.01.0.0.1. Deep Dive: Summarize, ELI5, Connect

## 09.02. Do Not Spam Unconfirmed Subscribers

##### 09.02.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
The **scoped mock** that is we create with `.mount_as_scoped` and name with `named` that is used in `create_unconfirmed_subscriber`.

_**Why?**_  
It seems like they help with ensuring that we drive the state of our application cleanly because they are local to the helpers they  
are created within.

_**Questions?**_  
None

##### 09.02.0.1. Deep Dive: Summarize, ELI5, Connect

_**Summary**_  
This summary covers up to $9.03$

Primarily 
- Add `newsletter_not_sent_to_unconfirmed_subscriber()` - Thing to note is that we expect 0 from our mock email server
- Add `create_unconfirmed_subscriber` - Thing to note here is we create a scoped mock/mount of the test `app.email_server`
- Add the `publish_newsletter` handler in newly created `src/routes/newsletter.rs` module.

Talk about scoped mocks and using public facing api already in place via the `subscribe` handler for `POST /subscriptions` that adds a new subscriber and  
clicking the confirmation link that confirms a subscriber

Let start by implementing `newsletter_are_not_delivered_to_unconfirmed_subscribers` and `create_unconfirmed_subscriber;
```Rust
//! src/api/main.rs
// [...]
// New test module

mod newsletter;

//! api/routes/newsletter.rs
use crate::common::{spawn_app, TestApp};
use wiremock::{ Mock, ResponseTemplate, matchers::{any, path, method}};

#[tokio::test]
async fn newsletters_not_delivered_to_unconfirmed_subscribers() {
    // Arrange
    let app = spawn_app().await;
    create_unconfirmed_subscriber(&app).await;
    
    // Act
    Mock::given(any())
        .respond_with(ResponseTemplate::new(200))
        // asserted at the end of the scope 
        // that no request is fired at Postmark
        .expect(0) 
        .mount(&app.email_server)
        .await;

    // A sketch of the newsletter payload structure
    let newsletter_request_body = serd_json::json!({
        "title": "Newsletter title",
        "content" : {
           "text": "Newsletter body as plain text",
            "html": "<p>Newsletter body as HTML</p>"
       }
    });

    let response = request::Client::new()
        .post(format!("{}/newsletters", &app.address))
        .json(newsletter_request_body)
        .send()
        .await
        .expect("Failed to execute newsletter post request");
    
    // Assert
    assert_eq!(response.status().as_u16(), 200);
    // Mock verifies on Drop that we havn't sent the newsletter email
}

/// Use the public API (via subscribe handler) of the application under test to
// create an unconfirmed subscriber
async fn create_unconfirmed_subscriber(app: &TestApp) {
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    let _mock_guard = Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .named("Mock email server for creating unconfirmed subscriber")
        .expect(1)
        .mount_as_scoped(&app.email_server)
        .await;
    
   app.post_subscriptions(body.into())
       .await
       .error_for_status()
       .expect("Failed to post subscription when creating unconfirmed user.");
}
```

<span style="color: orange; font-weight: bold">--INSERT SCREENSHOT HERE--</span>

This should return a 404 because we have not handled the `POST /newsletters` route.

To go green we just add the handler and update the `Router`.
```Rust
//! src/routes/mod.rs
// [...]
mod newsletter;

// [...]
pub use newsletter::*;

//! src/routes/newsletter.rs
use actix_web::HttpResponse;

#[tracing::instrument(
    name = "Publish newsletter"
)]
pub async fn publish_newsletter() -> HttpResponse {
    HttpResponse::Ok().finish()
}

//! src/ startup.rs
// [...]
use crate::routes{/**/, publish_newsletter};

fn run(/**/) -> Result<Server, std::io::Error> {
    // [...]
    let server = HttpServer::new(move || {
        App::new()
            .wrap(TracingLogger::default())
            // Registering the new handler
            .route("/newsletters", web::post().to(post_newsletter))
            // [...]
    })
    // [...]
}
```

### 9.02.0 Overview

### 9.02.1. Set Up State Using The Public API

### 9.02.3. Scoped Mocks

### 9.02.4 Green Test

## 09.03. All Confirmed Subscribers Receive New Issues

### 09.03.1. Composing Test Helpers

##### 09.03.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
How we modify `creat_unconfirmed_subscriber()` and reuse it in `create_confirmed_subscriber`

_**Why?**_  
Just a small clever change enables us to drive the application state without code duplication.

_**Questions**_
None

##### 09.03.0.1. Deep Dive: Summarize, ELI5, Connect

_**Summary**_  

Primarily 
- Update our implementation of `create_unconfirmed_subscriber` by returning confirmation links in order to reuse with `create_confirmed_subscriber`
- Add `newsletter_sent_to_confirmed_subscriber` test

## 09.04. Implementation Strategy

##### 09.04.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
Simple naive approach that iterates through all confirmed subscribers and sends an email with newsletter issue

_**Why?**_  
Simplicity of the the naive strategy

##### 09.04.0.1. Deep Dive: Summarize, ELI5, Connect

## 09.05. Body Schema

### 09.05.0. Overview

##### 09.05.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
Using the appropriate `actix_web` extractor ensures that our request is validated automaitically.  
In this case we use `actix_web::web::Json` with the an expected `BodyData` shape. This ensures that our
`post_newsletter` handler has valid json.

_**Why?**_  
In a previous chapter using `actix_web::web::Query` ensured that we had the appropriate query parameters for our  
`confirm handler. Type system ensuring we always have the valid data for each request making testing easy.

_**Questions?**_  
None

##### 09.05.0.1. Deep Dive: Summarize, ELI5, Connect

### 09.05.1. Testing Invalid Inputs

## 09.06. Fetch Confirmed Subscribers

##### 09.06.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
`query_as!` that allows us to map a returned SQL to a type we specify as the first argument. In this case `ConfirmedSubscriber`.

Also notice that for the custom error we define `PublishError` we only enumerate 1 variant `UnexpectedError` for a select statement  
where we return all confirmed subscribers. The status we return is `StatusCode::INTERNAL_SERVER_ERROR` which I wonder if it is a  
good response for rows not found. Will  investigate further later on, what would be more appropriate here.

Also noticed the reuse of `error_chain_fmt`. Might have been the case that `anyhow`'s debug implementation did not have the `Caused by`  
error source chaining yet.

_**Why?**_  
- `query_as!` for the ergonomics that it provides
- `Unexpected` internal server error because I encountered the same choice when deciding what to return for the `confirm` handler
  if unable to get a subscriber id for a given `subscription_token`.

_**Questions?**_  
- Maybe check when `anyhow` updated their `Debug` implementation to include error source chaining with `Cause by` substring.
  > Seems to be the [case](https://docs.rs/anyhow/latest/anyhow/struct.Error.html#display-representations)

##### 09.06.0.0.1. Deep Dive: Summarize, ELI5, Connect

## 09.07. Send Newsletter Emails

### 09.07.0. Overview

##### 09.07.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
`anyhow`'s `with_context` that we use in the `send_event` failure mode, 

_**Why?**_  
Because we are sending emails in an iteration we want to only do error string allocation in when an error is encountered.


_**Question?**_  
None

##### 09.07.0.1. Deep Dive: Summarize, ELI5, Connect

### 09.07.1. `context` vs `with_context`

## 09.08. Validation Of Stored Data

### 09.08.0. Overview

##### 09.08.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
In a high frequency deployment pipeline data stored in a DB can lead to _coupling old version of and new versions of an application_.  
In this case we might have _"valid email"_ by a prior criteria that has been updated to a more stricter validation protocal making some 
previous subscriber emails invalid.  

Use of `filter_map` in `get_confirmed_subscribers`

_**Why?**_  
This is an important consideration and is highlighted here when validating at insertion and validating at query time for applying business
logic to the data.

Great real world practical example of how `filter_map` would be used especially with a db query into a custom type.

_**Questions?**_  


##### 09.08.0.1. Deep Dive: Summarize, ELI5, Connect

### 09.08.1. Responsibility Boundaries

##### 09.08.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
Separation of concerns between were we call out that `get_confirmed_subscriber` is actually an **adapter** between the data layer and  
domain layer. Therefor it is inappropriate to apply filtering business logic and logging.  
This instead is the `publish_newsletter` handlers responsibility.

Tied to this is how we then update `get_confirmed_subscriber` return type to `Result<Vec<Result<ConfirmedSubscriber>, anyhow::Error>, anyhow::Error>`

_**Why?**_  
It is interesting to follow the authors reasoning as to why this is the preferable approach. And how this pattern generalizes to common architectural  
patterns (MVC, MVVM, MVP)

_**Questions?**_  

##### 09.08.0.1. Deep Dive: Summarize, ELI5, Connect

### 09.08.2.Follow The Compiler

##### 09.08.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
How we update `send_email` signature to accept a reference to `SubscriberEmail` as opposed to owning it.

_**Why?**_  
Always wondered why we did not use a reference here in the first place

_**Questions?**_  
Need to understand what `as_ref()` really does compared to `&example`?
> Some thing about how `as_ref()` allows us to be generic over the inner type alternatives. Explored [here](https://gemini.google.com/share/cf7f4fa3b009)

### 09.08.3. Remove Come Boiler Plate

##### 09.08.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
Completing removing `struct Row {email: String}` from `get_subscribe_email` and reverting to using `sqlx::query!`

_**Why?**_  
Wondering if it was necessary to go through all that to come back to `sqlx::query!`. But at least we know how to achieve
a query and type maping at the same time.

_**Questions?**_  
None

## 09.09. Limiitations Of Naive Approach

##### 09.09.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
The list of shortcomings. That one that stands out to me are
1. Security
2. Performance
3. Fault Tolerance

_**Why?**_  
Want to understand the Security and Fault Tolerance pieces in order to better scaffold a start axum api project.

_**Questions?**_  
None

##### 09.09.0.1. Deep Dive: Summarize, ELI5, Connect

## 09.10. Summary

##### 09.10.0.0. Skimming: What did you notice and why? Any Questinos

_**What?**_  
This chapter is very palletable in a single sitting and not as information dense which I appreciate.

_**Why?**_  
Would be nice to see how to divide the next _Securing Our API_ into bite sized chunks that are more digestable  
because it is a long and information dense chapter.


_**Questions?**_  

##### 09.10.0.1. Deep Dive: Summarize, ELI5, Connect